# MATH 105 — Wildfire Neural Network Lab
### Is a bigger network better? What if we change the decision rule?

**Goal:** Work with the actual wildfire observations. The network predicts whether an **observed fire** is **High/Extreme** (`1`) or **Low/Moderate** (`0`).

You already know how a neural network calculates a prediction. Today you'll use Python to train one, test whether a more complicated network helps, and decide how to use its predicted probabilities to issue alerts.

## 1. Train the original network (10 minutes)

Our starting model has **four weather inputs** (temperature, humidity, wind, precipitation), **one hidden layer of three sigmoid neurons**, and **one output**. The code prepares an 80% training set and a 20% testing set. It keeps the same split for every experiment.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, log_loss

# Our data and labels
df = pd.read_csv("wildfire_multi_region_dataset.csv")
features = ["temp_max_c", "humidity_pct", "wind_max_kmh", "precip_mm"]
X = df[features].copy()
y = df["fire_intensity"].isin(["High", "Extreme"]).astype(int)

# Same input scaling used in our earlier activity
X = X / [50, 100, 50, 10]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
y_test = y_test.to_numpy()  # Easier to compare with predicted labels

# Python trains all the weights and biases for us.
def fit_network(hidden_layers):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation="logistic",  # Sigmoid hidden neurons
        solver="adam",         # Gradient-based optimization
        learning_rate_init=0.01,
        max_iter=600,
        n_iter_no_change=30,
        tol=1e-6,
        random_state=42
    )
    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_test)[:, 1]
    return model, probabilities

# Train the original 4 -> 3 -> 1 network.
original_model, probabilities = fit_network((3,))

# Summarize how the network performs on data it has not seen.
original_predictions = (probabilities >= 0.5).astype(int)
original_parameters = (sum(w.size for w in original_model.coefs_) +
                       sum(b.size for b in original_model.intercepts_))
original_missed = np.sum((y_test == 1) & (original_predictions == 0))

original_results = pd.DataFrame([{
    "Model": "Original Model",
    "Parameters": original_parameters,
    "Test accuracy": accuracy_score(y_test, original_predictions),
    "Test cross-entropy": log_loss(y_test, probabilities),
    "High/Extreme missed": original_missed
}])
display(original_results.round(4))

### Understanding the Model Results

The table summarizes how well our neural network performs on the **testing data**—fire observations it was **not trained on**.

- **Model:** The name of the network we trained.
- **Parameters:** The total number of weights and biases learned during training. Our 4-input, 3-hidden-neuron, 1-output network has **19 parameters**.
- **Test accuracy:** The proportion of testing observations classified correctly. About **60%** are classified correctly using a probability threshold of 0.5.
- **Test cross-entropy:** The average penalty for the probabilities the network assigns to the actual labels. **Lower is better.** Unlike accuracy, it accounts for *how strongly* the network predicts a class, not just whether its classification is right.
- **High/Extreme missed:** The number of actual High/Extreme fires classified as Low/Moderate. This model misses **roughly 1,100 of the 1,229** High/Extreme fires in the testing data.

**Think about it:** A model can classify about 60% of observations correctly and still miss most of the fires we particularly want to identify. Does 60% test accuracy tell the whole story?

## 2. Does making the network bigger help? (15 minutes)

**Make a prediction first:** Would adding **more neurons** or **another hidden layer** noticeably improve predictions? Why?

Train these three networks with the **same four weather inputs**:

- **Original Model:** one hidden layer with 3 neurons, `(3,)`.
- **More Neurons:** one hidden layer with 10 neurons, `(10,)`.
- **More Layers:** two hidden layers with 3 neurons each, `(3, 3)`.

**Your task:** Add the two missing models to `architectures`. Run the experiment, then explain what the results suggest.

In [ ]:
# Add two experiments to the list below.
# Format: ("Name", (neurons in hidden layer 1, ...))
architectures = [
    ("Original Model", (3,)),
    # TODO: add "More Neurons" with 10 hidden neurons
    # TODO: add "More Layers" with two hidden layers of 3 neurons each
]

architecture_results = []
for name, hidden_layers in architectures:
    model, probs = fit_network(hidden_layers)
    predictions = (probs >= 0.5).astype(int)
    missed = np.sum((y_test == 1) & (predictions == 0))
    parameters = (sum(w.size for w in model.coefs_) +
                  sum(b.size for b in model.intercepts_))

    architecture_results.append({
        "Network": name,
        "Hidden layers": str(hidden_layers),
        "Parameters": parameters,
        "Test accuracy": accuracy_score(y_test, predictions),
        "Test cross-entropy": log_loss(y_test, probs),
        "High/Extreme missed": missed
    })

display(pd.DataFrame(architecture_results).round(4))

**Interpret:** How much did the number of parameters increase? Did testing accuracy, cross-entropy, or missed intense fires improve *meaningfully*? Explain using your table.

**The researchers are disappointed:** The original network misses a large number of High/Extreme fires. Instead of adding more neurons, can we change how the organization **uses** the probabilities?

## 3. You're in Charge of Wildfire Alerts (20 minutes)

Your neural network predicts the **probability** that an observed fire belongs to the High/Extreme intensity group. So far, we've used a classification threshold of 0.5, i.e. if the predicted probability is at least 0.5, we classify the fire as High/Extreme.

But the wildfire monitoring organization has a problem! The network is missing many fires that actually have High or Extreme intensity.

The organization uses the network's predictions to decide which fires should receive additional attention. Unfortunately, there are two ways the network can make a mistake:

- **Missed intense fire:** The network predicts Low/Moderate, but the fire is actually High/Extreme. This could mean failing to investigate a potentially dangerous fire.
- **False alarm:** The network predicts High/Extreme, but the fire is actually Low/Moderate. This could mean spending limited resources investigating a fire that does not need the same level of attention.

**Both mistakes have consequences, but are they equally serious?**

The organization believes that missing an intense fire is more costly than issuing a false alarm. To account for this, it proposes assigning a **penalty** to each type of mistake:

- **5 penalty points** for every missed High/Extreme fire.
- **1 penalty point** for every false alarm.

The total penalty is

$$
P=5(\text{Missed Intense Fires})+(\text{False Alarms}).
$$

These penalty points are hypothetical; they represent how the organization values different mistakes, not actual dollar costs.

**Our goal is to find a classification threshold that minimizes the total penalty.**

Lowering the threshold might help us identify more intense fires, but it could also generate more false alarms. Increasing the threshold might reduce false alarms while causing us to miss more intense fires.

### Your Task

Experiment with different classification thresholds. For each threshold, calculate:

- **Missed intense fires:** Actual label `1`, predicted label `0`.
- **False alarms:** Actual label `0`, predicted label `1`.
- **Alerts issued:** All observations predicted as `1`.
- **Total penalty:** The penalty associated with the two types of mistakes.

Complete the four `TODO` lines below. You already have the network's testing probabilities stored in `probabilities`. Use NumPy comparisons and `np.sum(...)`.

Once you've completed your calculations, compare the thresholds. **Which threshold would you recommend to the organization, and what tradeoff does your recommendation involve?**

In [ ]:
thresholds = [0.5]#you can add more thresholds to test here
threshold_results = []

for threshold in thresholds:
    # TODO 1: Predict 1 if probability >= threshold, and 0 otherwise.
    predicted = (probabilities >= threshold).astype(int)

    # TODO 2: Count actual High/Extreme fires predicted Low/Moderate.
    missed = np.sum((y_test == ???) & (predicted == ???))

    # TODO 3: Count actual Low/Moderate fires predicted High/Extreme.
    false_alarms = np.sum((y_test == ???) & (predicted == ???))

    if predicted is None or missed is None or false_alarms is None:
        raise ValueError("Complete the three TODO calculations above.")

    alerts = np.sum(predicted == 1)

    # Planning penalty: 2 points for a missed intense fire,
    # and 1 point for a false alarm. (These are illustrative.)
    # TODO 4: Calculate the total penalty.
    penalty = 5 * missed + false_alarms
    if penalty is None:
        raise ValueError("Complete the penalty calculation.")

    threshold_results.append({
        "Threshold": threshold,
        "High/Extreme missed": missed,
        "False alarms": false_alarms,
        "Alerts issued": alerts,
        "Penalty": penalty,
        "Accuracy": accuracy_score(y_test, predicted)
    })

threshold_table = pd.DataFrame(threshold_results)
display(threshold_table.round(4))

### Make a recommendation

Answer using the table you calculated.

**1. Tradeoff:** As you lower the threshold, what happens to the number of High/Extreme fires missed? What happens to false alarms? Why?

**2. Which threshold in your table gives the *lowest total penalty*? Is it the threshold with the highest accuracy?

**3. Breaking news!** The organization has enough staff to investigate **at most 800 alerts** from these 3,100 observations. Which thresholds meet this constraint? Among those, which one would you recommend if you want to miss as few intense fires as possible? Support your recommendation with numbers.

**4. What is limiting the model?** What did changing the architecture accomplish compared with changing the threshold? Does changing the threshold improve the neural network's predicted probabilities or its cross-entropy?

### Exit Ticket: What Have We Learned About Neural Networks?

1. When we use `model.fit()` to train a neural network, what is Python actually doing? In your answer, explain the role of the **weights, biases, gradients, and cost function**.

2. We experimented with adding more hidden neurons and another hidden layer. Did these changes improve our model's performance? What does this suggest about the relationship between model complexity and predictive accuracy?

3. We also experimented with changing the classification threshold. **Did changing the threshold change the neural network itself?** Explain what changed and why that mattered for our wildfire monitoring organization.

4. Based on your investigations, what do you think is the biggest limitation of our wildfire classifier? What might we need to make better predictions?